---
---
# Universidad Federico Santa María - 2026

<img src="https://fdiaz1968.github.io/Finance-MBA/images/logo_utfsm.png" alt="Universidad Técnica Federico Santa María - Departamento de Ingeniería Comercial" style="width: 480px !important; max-width: 100% !important; height: auto !important;"/>

## FINANZAS

### Profesor Fernando Díaz H.
---

# 📐 Estimación de Betas: El Modelo de Mercado

El **beta** ($\beta$) de una acción mide cuánto se mueve, en promedio, su retorno cuando se mueve el mercado. Es la medida de **riesgo sistemático** que utiliza el CAPM y, por lo tanto, un insumo central para calcular el costo del capital propio de una empresa.

En la práctica, el beta no se observa: se **estima**. La forma estándar de hacerlo es el **modelo de mercado**, una regresión lineal simple entre el retorno de la acción y el retorno del mercado:

$$r_{i,t}=\alpha_{i}+\beta_{i}\,r_{M,t}+\varepsilon_{i,t}$$

donde:

* $r_{i,t}$ es el retorno logarítmico mensual de la acción $i$ en el mes $t$;
* $r_{M,t}$ es el retorno logarítmico mensual del mercado, aproximado por el índice **S&P 500**;
* $\alpha_{i}$ es el intercepto de la regresión;
* $\beta_{i}$ es la pendiente, es decir, el **beta** de la acción;
* $\varepsilon_{i,t}$ es el residuo: la parte del retorno que **no** se explica por el mercado.

En este notebook:

1. Descargaremos precios históricos de cinco acciones —AAPL, WMT, INTC, XOM y LMT— y del S&P 500.
2. Calcularemos sus retornos logarítmicos **mensuales**.
3. Visualizaremos la relación entre cada acción y el mercado.
4. Estimaremos el modelo de mercado para cada acción y construiremos una tabla resumen de betas, con sus errores estándar e intervalos de confianza.
5. Descompondremos el riesgo de cada acción en **sistemático** y **específico**.
6. Usaremos los betas estimados para calcular retornos esperados con el **CAPM** y graficar la Línea de Mercado de Valores (SML).
7. Repetiremos la estimación con retornos **en exceso** de la tasa libre de riesgo, para obtener el **alfa de Jensen**.

> 💡 **Idea central:** el beta es la pendiente de la regresión, pero también puede escribirse como
>
> $$\beta_{i}=\frac{\operatorname{Cov}\left(r_{i},r_{M}\right)}{\operatorname{Var}\left(r_{M}\right)}$$
>
> Es decir, mide cuánto **co-varía** la acción con el mercado, en relación con la variabilidad propia del mercado. Además, el $R^{2}$ de la regresión indica qué fracción del riesgo de la acción es sistemático.

## 📦 Cargando las librerías

Antes de comenzar el análisis, cargaremos los paquetes de **R** que utilizaremos a lo largo del notebook:

* **`tidyquant`**: descarga y manipulación de datos financieros.
* **`tidyr`**: reorganización de datos entre formatos ancho y largo.
* **`dplyr`**: transformación, filtrado y organización de bases de datos.
* **`ggplot2`**: construcción de gráficos.
* **`scales`**: formato de los ejes (porcentajes).
* **`broom`**: conversión de los resultados de una regresión a tablas ordenadas (*tibbles*).
* **`stargazer`**: tablas de regresión con formato académico.

También ajustaremos el tamaño y la resolución de los gráficos para mejorar su visualización dentro del notebook.

> 💡 Las librerías deben instalarse una sola vez, pero deben cargarse cada vez que se inicia una nueva sesión de R.

In [ ]:
#install.packages(c("tidyquant", "broom", "stargazer"))

In [ ]:
suppressWarnings(suppressPackageStartupMessages({
  library(tidyquant) # To download the data
  library(tidyr)
  library(dplyr)
  library(ggplot2)
  library(scales)
  library(broom)
  library(stargazer)
}))

# Bigger, higher-resolution plots when rendered to the web page
options(repr.plot.width = 10, repr.plot.height = 6, repr.plot.res = 150)

---
## 📥 Extracción de datos bursátiles

Estimaremos los betas de las mismas cinco acciones que utilizamos en el notebook de portafolios, más el índice que usaremos como aproximación del mercado:

* **Apple Inc. (`AAPL`)**: tecnología.
* **Walmart Inc. (`WMT`)**: comercio minorista.
* **Intel Corporation (`INTC`)**: semiconductores.
* **Exxon Mobil Corporation (`XOM`)**: energía.
* **Lockheed Martin Corporation (`LMT`)**: aeroespacial y defensa.
* **S&P 500 (`^GSPC`)**: índice de mercado.

### 🏷️ Definiendo los *tickers*

Guardaremos las acciones en el vector `tick` y el símbolo del mercado en un objeto separado, `mercado`, porque en la regresión cumplen roles distintos: las acciones son las variables **dependientes** y el mercado es la variable **explicativa**.

In [ ]:
tick <- c('AAPL', 'WMT', 'INTC', 'XOM', 'LMT')
mercado <- '^GSPC'

tick

### 📥 Descarga de precios históricos

Descargaremos cinco años de precios diarios: enero de 2021 a diciembre de 2025, lo que nos dará **60 retornos mensuales**.

Comenzamos en **diciembre de 2020** para disponer del precio de cierre de ese mes, que es el precio base para calcular el retorno de enero de 2021.

> 💡 La columna `adjusted` contiene el precio ajustado por dividendos y splits, por lo que los retornos calculados incorporan ambos efectos.

In [ ]:
price_data <- tq_get(c(tick, mercado),
                     from = '2020-12-01',
                     to = '2026-01-01',
                     get = 'stock.prices')

print(price_data)

---
## 📈 Retornos logarítmicos mensuales

Para estimar betas se acostumbra usar retornos **mensuales** (o semanales), porque los retornos diarios pueden distorsionarse por efectos de sincronización en la negociación de las acciones. Una práctica habitual es usar entre 3 y 5 años de retornos mensuales.

Con `tq_transmute()` y `periodReturn` calculamos el retorno logarítmico mensual de cada símbolo, agrupando por `symbol`:

$$r_{i,t}=\ln\left(\frac{P_{i,t}}{P_{i,t-1}}\right)$$

donde $P_{i,t}$ es el precio de cierre ajustado de la acción $i$ al final del mes $t$.

> 💡 `periodReturn` también entrega un retorno "parcial" para el primer mes de la muestra (diciembre de 2020), calculado desde el primer día disponible. Como no es un mes completo, lo descartamos con `filter()`.

In [ ]:
log_ret_tidy <- price_data |>
  group_by(symbol) |>
  tq_transmute(select = adjusted,
               mutate_fun = periodReturn,
               period = 'monthly',
               col_rename = 'ret',
               type = 'log') |>
  ungroup() |>
  filter(date >= as.Date('2021-01-01'))

head(log_ret_tidy)

### 🔄 Transformación a formato ancho

Para estimar las regresiones necesitamos una columna por serie y una fila por mes. Pasamos de formato largo a ancho con `pivot_wider()` y, de paso, renombramos el índice `^GSPC` como `SP500`, para que las regresiones sean más fáciles de leer.

In [ ]:
log_ret <- log_ret_tidy |>
  mutate(symbol = if_else(symbol == '^GSPC', 'SP500', symbol)) |>
  pivot_wider(names_from = symbol, values_from = ret) |>
  drop_na()

cat(sprintf("Observaciones mensuales: %d\n", nrow(log_ret)))
head(log_ret)

## 📊 Estadística descriptiva

Antes de estimar, conviene mirar los datos. La siguiente tabla (`stargazer`) muestra estadísticos descriptivos de los retornos logarítmicos mensuales de cada serie.

In [ ]:
stargazer(as.data.frame(select(log_ret, -date)), type = "text",
          title = "Estadística Descriptiva", digits = 4)

# Para guardar la tabla en un archivo:
# stargazer(as.data.frame(select(log_ret, -date)), type = "text",
#           title = "Estadística Descriptiva", digits = 4,
#           out = "Estadistica Descriptiva.txt")

---
## 🔎 Visualizando la relación: acción vs. mercado

Antes de correr las regresiones, graficaremos para cada acción sus retornos mensuales (eje vertical) contra los del S&P 500 (eje horizontal). Cada punto es un mes.

La recta roja es el ajuste por mínimos cuadrados: su **pendiente es el beta** de la acción. Esta recta se conoce como la **línea característica** de la acción.

In [ ]:
log_ret |>
  pivot_longer(cols = all_of(tick), names_to = "Empresa", values_to = "ret_accion") |>
  ggplot(aes(x = SP500, y = ret_accion)) +
  geom_hline(yintercept = 0, color = "gray", linetype = "dotted") +
  geom_vline(xintercept = 0, color = "gray", linetype = "dotted") +
  geom_point(color = "steelblue", size = 2, alpha = 0.8) +
  geom_smooth(method = "lm", formula = y ~ x, se = FALSE, color = "#d62728", linewidth = 1) +
  facet_wrap(~ Empresa, ncol = 3) +
  scale_x_continuous(labels = percent_format(accuracy = 1)) +
  scale_y_continuous(labels = percent_format(accuracy = 1)) +
  labs(x = "Retorno mensual S&P 500", y = "Retorno mensual de la acción",
       title = "Línea característica de cada acción") +
  theme_minimal()

---
## 🧮 Estimación del modelo de mercado

Estimaremos, para cada acción $i$, la regresión:

$$r_{i,t}=\alpha_{i}+\beta_{i}\,r_{M,t}+\varepsilon_{i,t}$$

mediante **mínimos cuadrados ordinarios (MCO)** con la función `lm()`. En la fórmula `AAPL ~ SP500`, el intercepto se incluye automáticamente.

### 🍎 Un ejemplo detallado: Apple

Comencemos con una sola acción para leer con calma la salida de la regresión.

In [ ]:
modelo_aapl <- lm(AAPL ~ SP500, data = log_ret)
summary(modelo_aapl)

### 📖 ¿Cómo leer esta salida?

En la tabla `Coefficients`, las filas relevantes son:

* **`(Intercept)`**: el intercepto $\hat{\alpha}$. Es el retorno mensual promedio de la acción que **no** se explica por el mercado. Bajo el CAPM debería ser cercano a cero.
* **`SP500`**: la pendiente $\hat{\beta}$, es decir, el beta estimado.

Las columnas que acompañan a cada coeficiente son:

* **`Std. Error`**: el error estándar de la estimación. Mide la **incertidumbre** del coeficiente.
* **`t value`** y **`Pr(>|t|)`**: el estadístico $t=\hat{\beta}/\text{error estándar}$ y su valor-$p$, para contrastar si el coeficiente es distinto de cero.

Y, más abajo, el **`Multiple R-squared`** ($R^{2}$): la fracción de la varianza del retorno de la acción que es explicada por el mercado.

> ⚠️ Un beta estimado **no** es el beta verdadero: es una estimación con error. Fíjese en el error estándar (y en el intervalo de confianza que calcularemos más adelante) antes de tomar decisiones con un beta "puntual".

In [ ]:
beta_aapl <- coef(modelo_aapl)[["SP500"]]
r2_aapl <- summary(modelo_aapl)$r.squared

cat(sprintf("Si el S&P 500 sube 1%% en un mes, AAPL tiende a subir %.2f%% (en promedio).\n", beta_aapl))
cat(sprintf("El mercado explica el %.1f%% de la varianza de los retornos mensuales de AAPL.\n", r2_aapl * 100))

### 🔁 Estimando las cinco acciones

Repetimos la estimación para las otras cuatro acciones y guardamos los cinco modelos en una **lista con nombre**, cuyas etiquetas son los símbolos. Así podremos recorrer los modelos fácilmente más adelante.

Primero mostramos los cinco modelos lado a lado con `stargazer`, como se acostumbra en un artículo académico:

In [ ]:
modelo_wmt  <- lm(WMT  ~ SP500, data = log_ret)
modelo_intc <- lm(INTC ~ SP500, data = log_ret)
modelo_xom  <- lm(XOM  ~ SP500, data = log_ret)
modelo_lmt  <- lm(LMT  ~ SP500, data = log_ret)

modelos <- list(AAPL = modelo_aapl, WMT = modelo_wmt, INTC = modelo_intc,
                XOM = modelo_xom, LMT = modelo_lmt)

stargazer(modelos, type = "text", title = "Modelo de Mercado", align = TRUE)

# Para guardar la tabla en un archivo HTML:
# stargazer(modelos, type = "html", title = "Modelo de Mercado", align = TRUE,
#           out = "MARKET MODEL.html")

Luego armamos una **tabla resumen** con `broom`, que incluye:

* el intercepto $\hat{\alpha}$;
* el beta $\hat{\beta}$, su error estándar, su estadístico $t$ y su intervalo de confianza al 95%;
* el $R^{2}$ y el número de observaciones.

In [ ]:
tabla_betas <- bind_rows(lapply(tick, function(a) {
  m <- modelos[[a]]
  b <- tidy(m, conf.int = TRUE) |> filter(term == "SP500")
  tibble(Accion = a,
         Alfa = coef(m)[["(Intercept)"]],
         Beta = b$estimate,
         Error_estandar = b$std.error,
         t_beta = b$statistic,
         IC_inf = b$conf.low,
         IC_sup = b$conf.high,
         R2 = glance(m)$r.squared,
         Obs = nobs(m))
}))

tabla_betas |> mutate(across(where(is.numeric), ~ round(.x, 4)))

### ✅ Verificación: el beta como $\operatorname{Cov}/\operatorname{Var}$

Recordemos que el beta de MCO coincide con la covarianza entre la acción y el mercado dividida por la varianza del mercado. Verifiquemos esta igualdad con los datos:

In [ ]:
beta_cov_var <- sapply(tick, function(a) cov(log_ret[[a]], log_ret$SP500) / var(log_ret$SP500))

comparacion <- tibble(Accion = tick,
                      Beta_MCO = tabla_betas$Beta,
                      Beta_Cov_Var = unname(beta_cov_var))
print(comparacion)

cat("¿Coinciden?", isTRUE(all.equal(comparacion$Beta_MCO, comparacion$Beta_Cov_Var)), "\n")

---
## 📊 Comparando los betas

Grafiquemos los betas estimados junto con su **intervalo de confianza al 95%**. La línea punteada marca $\beta=1$, el beta del mercado en su conjunto:

* $\beta>1$: acción **agresiva**, amplifica los movimientos del mercado.
* $\beta<1$: acción **defensiva**, los atenúa.

In [ ]:
tabla_betas |>
  mutate(Tipo = if_else(Beta > 1, "Agresiva (β > 1)", "Defensiva (β < 1)")) |>
  ggplot(aes(x = reorder(Accion, Beta), y = Beta, fill = Tipo)) +
  geom_col(color = "black") +
  geom_errorbar(aes(ymin = IC_inf, ymax = IC_sup), width = 0.2) +
  geom_text(aes(y = IC_sup, label = sprintf("%.2f", Beta)), vjust = -0.6, fontface = "bold") +
  geom_hline(yintercept = 1, linetype = "dashed", color = "gray40") +
  scale_fill_manual(values = c("Agresiva (β > 1)" = "#d62728", "Defensiva (β < 1)" = "steelblue")) +
  labs(x = NULL, y = "Beta estimado",
       title = "Betas estimados con intervalo de confianza al 95%") +
  theme_minimal() +
  theme(legend.position = "bottom", legend.title = element_blank())

## 🧩 Riesgo sistemático vs. riesgo específico

La regresión permite descomponer la varianza del retorno de cada acción en dos partes:

$$\operatorname{Var}\left(r_{i}\right)=\underbrace{\beta_{i}^{2}\operatorname{Var}\left(r_{M}\right)}_{\text{riesgo sistemático}}+\underbrace{\operatorname{Var}\left(\varepsilon_{i}\right)}_{\text{riesgo específico}}$$

Dividiendo por $\operatorname{Var}\left(r_{i}\right)$, la fracción sistemática es exactamente el $R^{2}$ de la regresión, y la fracción específica es $1-R^{2}$.

* El **riesgo sistemático** depende del mercado y **no se puede diversificar**; es el que el CAPM remunera.
* El **riesgo específico** es propio de la empresa y **sí se puede diversificar** al combinar varias acciones en un portafolio; por eso el mercado no lo remunera.

In [ ]:
tabla_betas |>
  transmute(Accion,
            `Sistemático (R²)` = R2,
            `Específico (1 − R²)` = 1 - R2) |>
  pivot_longer(-Accion, names_to = "Tipo", values_to = "Fraccion") |>
  mutate(Tipo = factor(Tipo, levels = c("Específico (1 − R²)", "Sistemático (R²)"))) |>
  ggplot(aes(x = reorder(Accion, Fraccion * (Tipo == "Sistemático (R²)")), y = Fraccion, fill = Tipo)) +
  geom_col(color = "black") +
  geom_text(aes(label = percent(Fraccion, accuracy = 1)),
            position = position_stack(vjust = 0.5)) +
  coord_flip() +
  scale_y_continuous(labels = percent_format(accuracy = 1)) +
  scale_fill_manual(values = c("Sistemático (R²)" = "steelblue", "Específico (1 − R²)" = "lightgray")) +
  labs(x = NULL, y = "Fracción de la varianza del retorno",
       title = "Descomposición del riesgo de cada acción") +
  theme_minimal() +
  theme(legend.position = "bottom", legend.title = element_blank())

---
## 🧭 Del beta al retorno esperado: el CAPM

Ahora usaremos los betas estimados para calcular el **retorno esperado** de cada acción según el CAPM:

$$E\left[r_{i}\right]=r_{f}+\beta_{i}\left(E\left[r_{M}\right]-r_{f}\right)$$

Necesitamos dos insumos adicionales:

* La tasa libre de riesgo $r_{f}$: al igual que en el notebook de portafolios, usaremos el rendimiento del **Treasury a 10 años** (serie `DGS10` de FRED).
* La **prima de riesgo de mercado** $E\left[r_{M}\right]-r_{f}$. No la estimaremos con esta muestra, porque una prima calculada con cinco años de datos es extremadamente ruidosa. En su lugar, la fijamos como un **supuesto** que usted puede modificar.

> ⚠️ El valor de `prima_mercado` es un supuesto ilustrativo. Pruebe distintos valores y observe cómo cambia la pendiente de la SML y los retornos esperados.

In [ ]:
# Tasa libre de riesgo: Treasury a 10 años, obtenida directamente desde FRED
rf_data <- read.csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10")
rf_data$DGS10 <- suppressWarnings(as.numeric(rf_data$DGS10))
rf_data <- rf_data[!is.na(rf_data$DGS10), ]

rf <- tail(rf_data$DGS10, 1) / 100  # última tasa disponible, convertida a decimal
cat(sprintf("Tasa libre de riesgo (Treasury 10 años, FRED DGS10, %s): %.2f%%\n",
            tail(rf_data$observation_date, 1), rf * 100))

prima_mercado <- 0.05   # SUPUESTO: prima de riesgo de mercado (5% anual)

capm <- tabla_betas |>
  select(Accion, Beta) |>
  mutate(Retorno_CAPM = rf + Beta * prima_mercado)

capm |> mutate(Beta = round(Beta, 3), Retorno_CAPM = percent(Retorno_CAPM, accuracy = 0.01))

### 📈 La Línea de Mercado de Valores (SML)

Graficamos cada acción según su beta (eje horizontal) y su retorno esperado CAPM (eje vertical). Por construcción, todas quedan **sobre la recta**: la SML tiene intercepto $r_{f}$ y pendiente igual a la prima de riesgo de mercado.

In [ ]:
puntos_ref <- tibble(Etiqueta = c("Tasa libre de riesgo", "Mercado (β = 1)"),
                     Beta = c(0, 1),
                     Retorno = c(rf, rf + prima_mercado))

ggplot() +
  geom_abline(intercept = rf, slope = prima_mercado, color = "#d62728", linewidth = 1) +
  geom_point(data = capm, aes(x = Beta, y = Retorno_CAPM), color = "steelblue", size = 3) +
  geom_text(data = capm, aes(x = Beta, y = Retorno_CAPM, label = Accion), vjust = -1) +
  geom_point(data = puntos_ref, aes(x = Beta, y = Retorno, shape = Etiqueta, fill = Etiqueta),
             size = 3.5, color = "black") +
  scale_shape_manual(values = c("Tasa libre de riesgo" = 22, "Mercado (β = 1)" = 23)) +
  scale_fill_manual(values = c("Tasa libre de riesgo" = "black", "Mercado (β = 1)" = "gold")) +
  scale_y_continuous(labels = percent_format(accuracy = 0.1)) +
  expand_limits(x = 0) +
  labs(x = "Beta", y = "Retorno esperado",
       title = "Línea de Mercado de Valores con los betas estimados") +
  theme_minimal() +
  theme(legend.position = "bottom", legend.title = element_blank())

---
## ⚠️ Precauciones metodológicas

Estimar un beta involucra decisiones que **cambian el resultado**. Antes de usar un beta en una valoración, tenga presente que:

* **El beta depende de la muestra.** Otra ventana de tiempo (por ejemplo, 2 años en vez de 5) o otra frecuencia (semanal o diaria en vez de mensual) entrega betas distintos.
* **El beta es una estimación con error.** Con 60 observaciones, los intervalos de confianza suelen ser amplios; dos acciones con betas puntuales distintos pueden ser estadísticamente indistinguibles.
* **El mercado es una aproximación.** El S&P 500 es solo un proxy de la cartera de mercado del CAPM, que en teoría incluye todos los activos riesgosos.
* **El beta cambia con el tiempo.** Cambios en el negocio, en el apalancamiento o en el ciclo económico modifican el riesgo sistemático de una empresa.
* **Retornos brutos vs. retornos en exceso.** Aquí regresamos retornos brutos, por simplicidad; la versión formal del CAPM utiliza retornos **en exceso** de la tasa libre de riesgo. Con datos mensuales, la diferencia en el beta suele ser pequeña, pero el intercepto cambia de significado: lo vemos en la última sección.
* **Ajuste de beta.** Es habitual "ajustar" los betas históricos hacia 1 (por ejemplo, $\beta_{aj}=\tfrac{2}{3}\hat{\beta}+\tfrac{1}{3}$), pues los betas tienden a acercarse a 1 con el tiempo.

### 🧭 Conclusión

El modelo de mercado permite estimar el beta de una acción como la pendiente de la regresión de sus retornos contra los del mercado, y entrega además su error estándar, un intervalo de confianza y el $R^{2}$, que separa el riesgo sistemático del específico.

Combinado con una tasa libre de riesgo y una prima de mercado, el beta permite calcular el retorno esperado de la acción con el CAPM, es decir, el **costo del capital propio** que exigen los inversionistas.

> 💡 En el notebook **Beta y Retorno Esperado** contrastamos, con las estimaciones de los grupos del curso, si los betas y los retornos esperados presentan la relación positiva que predice la SML.

---
# 🏆 Retornos en exceso y el alfa de Jensen

En las secciones anteriores estimamos el modelo de mercado con retornos **brutos**. Sin embargo, el CAPM no está escrito en términos de retornos, sino de **primas de riesgo**, es decir, de retornos **en exceso** de la tasa libre de riesgo:

$$E\left[r_{i}\right]-r_{f}=\beta_{i}\left(E\left[r_{M}\right]-r_{f}\right)$$

Para llevar esta ecuación a los datos, definimos el retorno en exceso de la acción y del mercado en cada mes $t$:

$$R_{i,t}=r_{i,t}-r_{f,t}\qquad\qquad R_{M,t}=r_{M,t}-r_{f,t}$$

y estimamos la regresión:

$$R_{i,t}=\alpha_{i}+\beta_{i}\,R_{M,t}+\varepsilon_{i,t}$$

### ❓ ¿Por qué la constante de esta regresión es el alfa de Jensen?

Tomemos esperanza a ambos lados de la regresión. Como $E\left[\varepsilon_{i,t}\right]=0$:

$$E\left[R_{i}\right]=\alpha_{i}+\beta_{i}\,E\left[R_{M}\right]$$

El CAPM, en cambio, afirma que $E\left[R_{i}\right]=\beta_{i}\,E\left[R_{M}\right]$: el retorno en exceso esperado depende **solo** del riesgo sistemático. Comparando ambas expresiones, la constante es exactamente la diferencia entre lo observado y lo que exige el CAPM:

$$\alpha_{i}=\underbrace{E\left[R_{i}\right]}_{\text{retorno en exceso observado}}-\underbrace{\beta_{i}\,E\left[R_{M}\right]}_{\text{retorno en exceso exigido por el CAPM}}$$

Esta medida se conoce como el **alfa de Jensen** (Jensen, 1968), y se interpreta como el **retorno anormal ajustado por riesgo**:

* $\alpha_{i}>0$: la acción rindió **más** de lo que compensa su riesgo sistemático (queda **sobre** la SML);
* $\alpha_{i}<0$: rindió **menos** de lo que compensa su riesgo sistemático (queda **bajo** la SML);
* $\alpha_{i}=0$: rindió exactamente lo que predice el CAPM.

Por eso, el test $t$ de la constante contrasta $H_{0}:\alpha_{i}=0$, es decir, si la acción es consistente con el CAPM. Bajo el CAPM, **todos** los alfas deberían ser cero.

### ⚠️ ¿Por qué esto no ocurre con retornos brutos?

Si en la regresión en exceso reemplazamos $R_{i,t}=r_{i,t}-r_{f,t}$ y $R_{M,t}=r_{M,t}-r_{f,t}$, y despejamos $r_{i,t}$:

$$r_{i,t}=\underbrace{\alpha_{i}+\left(1-\beta_{i}\right)r_{f}}_{\text{intercepto con retornos brutos}}+\beta_{i}\,r_{M,t}+\varepsilon_{i,t}$$

Con retornos brutos, el intercepto **mezcla** el alfa con un término, $\left(1-\beta_{i}\right)r_{f}$, que depende de la tasa libre de riesgo y del beta. Solo coincide con el alfa de Jensen si $\beta_{i}=1$ o si $r_{f}=0$. Además, aun si el CAPM fuera cierto, ese intercepto **no** sería cero, sino $\left(1-\beta_{i}\right)r_{f}$; por lo tanto, contrastar "intercepto $=0$" con retornos brutos sería el test equivocado.

> 💡 **Idea central:** con retornos en exceso, la constante mide la **distancia vertical entre la acción y la SML**. Con retornos brutos, en cambio, la constante no tiene esa interpretación.

### 📥 Tasa libre de riesgo mensual

Necesitamos una tasa libre de riesgo con **frecuencia mensual**, para restarla a los retornos mensuales. Usaremos el rendimiento del **T-Bill a 3 meses** (serie `TB3MS` de FRED), pues su horizonte es el más cercano al de un retorno mensual.

La serie `TB3MS` está expresada como tasa anual en porcentaje. Para que sea comparable con nuestros retornos logarítmicos mensuales, la convertimos a un retorno logarítmico mensual:

$$r_{f,t}=\frac{\ln\left(1+y_{t}/100\right)}{12}$$

donde $y_{t}$ es la tasa anual (en %) del mes $t$.

> ⚠️ En la sección del CAPM usamos el Treasury a 10 años, porque allí necesitábamos una tasa de largo plazo para calcular un retorno esperado **anual**. Aquí, en cambio, necesitamos una tasa que sea comparable con retornos **mensuales**; por eso cambiamos de serie.

In [ ]:
# Tasa libre de riesgo mensual: T-Bill a 3 meses (FRED, serie TB3MS, % anual)
rf_raw <- read.csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=TB3MS")
rf_raw$TB3MS <- suppressWarnings(as.numeric(rf_raw$TB3MS))
rf_raw <- rf_raw[!is.na(rf_raw$TB3MS), ]

# Tasa anual (%) -> retorno logarítmico mensual
rf_mensual <- tibble(mes = format(as.Date(rf_raw$observation_date), "%Y-%m"),
                     rf  = log(1 + rf_raw$TB3MS / 100) / 12)

# Alineamos la serie con los meses de nuestros retornos
log_ret_rf <- log_ret |>
  mutate(mes = format(date, "%Y-%m")) |>
  left_join(rf_mensual, by = "mes")

if (anyNA(log_ret_rf$rf)) stop("Faltan datos de TB3MS para algunos meses de la muestra")

cat(sprintf("Tasa libre de riesgo mensual promedio: %.3f%%  (≈ %.2f%% anual)\n",
            mean(log_ret_rf$rf) * 100, mean(log_ret_rf$rf) * 12 * 100))

### ➖ Retornos en exceso

Restamos la tasa libre de riesgo mensual a los retornos de cada acción **y** a los del mercado:

In [ ]:
exceso <- log_ret_rf |>
  mutate(across(c(all_of(tick), SP500), ~ .x - rf)) |>
  select(date, all_of(tick), SP500)

head(exceso)

### 🔁 Estimando las cinco acciones

Estimamos la regresión en exceso para cada acción. La tabla resumen destaca ahora el **intercepto**, que es el alfa de Jensen, junto con su error estándar, su estadístico $t$ y su valor-$p$. Como los retornos son mensuales, anualizamos el alfa multiplicándolo por 12.

Para leer con calma la salida completa, primero mostramos el modelo de Apple: la fila `const` (Python) o `(Intercept)` (R) es el alfa de Jensen.

In [ ]:
modelo_exc_aapl <- lm(AAPL ~ SP500, data = exceso)
summary(modelo_exc_aapl)

In [ ]:
modelo_exc_wmt  <- lm(WMT  ~ SP500, data = exceso)
modelo_exc_intc <- lm(INTC ~ SP500, data = exceso)
modelo_exc_xom  <- lm(XOM  ~ SP500, data = exceso)
modelo_exc_lmt  <- lm(LMT  ~ SP500, data = exceso)

modelos_exc <- list(AAPL = modelo_exc_aapl, WMT = modelo_exc_wmt, INTC = modelo_exc_intc,
                    XOM = modelo_exc_xom, LMT = modelo_exc_lmt)

stargazer(modelos_exc, type = "text",
          title = "Modelo de Mercado con Retornos en Exceso (alfa de Jensen)", align = TRUE)

In [ ]:
tabla_jensen <- bind_rows(lapply(tick, function(a) {
  m  <- modelos_exc[[a]]
  td <- tidy(m, conf.int = TRUE)
  al <- filter(td, term == "(Intercept)")
  be <- filter(td, term == "SP500")
  tibble(Accion = a,
         Alfa_mensual = al$estimate,
         Alfa_anual = al$estimate * 12,
         Error_estandar = al$std.error,
         t_alfa = al$statistic,
         Valor_p = al$p.value,
         IC_inf_anual = al$conf.low * 12,
         IC_sup_anual = al$conf.high * 12,
         Beta = be$estimate,
         R2 = glance(m)$r.squared)
}))

tabla_jensen |> mutate(across(where(is.numeric), ~ round(.x, 4)))

### 📖 ¿Cómo leer esta tabla?

* **Alfa mensual / anualizado:** el retorno anormal promedio de la acción, después de descontar el retorno que exige su riesgo sistemático.
* **$t$ y valor-$p$ del alfa:** contrastan $H_{0}:\alpha_{i}=0$. Un valor-$p$ menor que 0,05 (o $|t|$ mayor que aproximadamente 2) es evidencia de que el alfa es distinto de cero.
* **Beta:** la pendiente en exceso, que debería ser similar al beta estimado con retornos brutos.

> ⚠️ Con 60 observaciones mensuales, el error estándar del alfa suele ser **grande** en relación con el propio alfa: un alfa llamativo en la tabla no es, por sí solo, evidencia de retornos anormales. Fíjese siempre en el intervalo de confianza.
>
> Además, rechazar $H_{0}:\alpha_{i}=0$ no prueba que la acción "gane más que el mercado": es una **hipótesis conjunta**. El rechazo puede deberse a que el CAPM está mal especificado, o a que el S&P 500 es un mal proxy de la cartera de mercado.

### 🔍 Comparación con la regresión con retornos brutos

Verifiquemos la relación que dedujimos: el intercepto de la regresión con retornos brutos debería ser, aproximadamente, el alfa de Jensen más $\left(1-\beta_{i}\right)\bar{r}_{f}$, donde $\bar{r}_{f}$ es la tasa libre de riesgo mensual promedio.

> 💡 La igualdad es **aproximada**, y no exacta, porque $r_{f}$ varía en el tiempo y porque los betas de ambas regresiones no son idénticos. Sería exacta si $r_{f}$ fuese constante.

In [ ]:
rf_prom <- mean(log_ret_rf$rf)

comparacion_alfa <- tibble(
  Accion = tick,
  Intercepto_brutos = tabla_betas$Alfa,
  Alfa_Jensen = tabla_jensen$Alfa_mensual,
  `(1 - beta) * rf promedio` = (1 - tabla_jensen$Beta) * rf_prom
) |>
  mutate(`Alfa Jensen + (1 - beta) * rf` = Alfa_Jensen + `(1 - beta) * rf promedio`,
         Beta_brutos = tabla_betas$Beta,
         Beta_exceso = tabla_jensen$Beta)

comparacion_alfa |> mutate(across(where(is.numeric), ~ round(.x, 4)))

### 📊 Alfa de Jensen anualizado con intervalo de confianza

Graficamos los alfas anualizados junto con su intervalo de confianza al 95%. Si el intervalo **incluye el cero**, no podemos rechazar $H_{0}:\alpha_{i}=0$: el retorno de la acción es consistente con el CAPM.

In [ ]:
tabla_jensen |>
  mutate(Signo = if_else(Alfa_anual > 0, "Alfa positivo", "Alfa negativo"),
         y_etiqueta = if_else(Alfa_anual >= 0, IC_sup_anual, IC_inf_anual),
         v_etiqueta = if_else(Alfa_anual >= 0, -0.6, 1.6)) |>
  ggplot(aes(x = reorder(Accion, Alfa_anual), y = Alfa_anual, fill = Signo)) +
  geom_col(color = "black") +
  geom_errorbar(aes(ymin = IC_inf_anual, ymax = IC_sup_anual), width = 0.2) +
  geom_text(aes(y = y_etiqueta, label = percent(Alfa_anual, accuracy = 0.1), vjust = v_etiqueta),
            fontface = "bold") +
  geom_hline(yintercept = 0) +
  scale_fill_manual(values = c("Alfa positivo" = "#2ca02c", "Alfa negativo" = "#d62728")) +
  scale_y_continuous(labels = percent_format(accuracy = 1), expand = expansion(mult = 0.12)) +
  labs(x = NULL, y = "Alfa de Jensen anualizado",
       title = "Alfa de Jensen con intervalo de confianza al 95%") +
  theme_minimal() +
  theme(legend.position = "bottom", legend.title = element_blank())

### 📈 El alfa de Jensen como distancia a la SML

Como la recta de mínimos cuadrados con intercepto pasa siempre por los promedios muestrales, el alfa estimado cumple exactamente:

$$\hat{\alpha}_{i}=\bar{R}_{i}-\hat{\beta}_{i}\,\bar{R}_{M}$$

Es decir, es la **distancia vertical** entre el retorno en exceso promedio de la acción y el punto que le correspondería sobre la **SML muestral**, una recta que parte en el origen y cuya pendiente es la prima de mercado observada en la muestra, $\bar{R}_{M}$.

Primero verificamos la igualdad y luego la graficamos. Las acciones **sobre** la recta tienen alfa positivo, y las que están **bajo** la recta, alfa negativo.

> ⚠️ Esta SML usa la prima de mercado **observada** en esta muestra (5 años), y no el supuesto de 5% de la sección del CAPM. Por eso, en este gráfico las acciones ya no quedan "sobre la recta por construcción": la distancia a la recta es justamente el alfa.

In [ ]:
prima_muestral <- mean(exceso$SP500) * 12    # prima de mercado observada (anualizada)

sml_muestral <- tabla_jensen |>
  select(Accion, Beta, Alfa_anual) |>
  mutate(Retorno_medio = unname(sapply(Accion, function(a) mean(exceso[[a]]) * 12)),
         Retorno_SML = Beta * prima_muestral)

cat("¿Alfa de Jensen = retorno en exceso medio − beta x prima de mercado observada?",
    isTRUE(all.equal(sml_muestral$Retorno_medio - sml_muestral$Retorno_SML, sml_muestral$Alfa_anual)), "\n")

In [ ]:
ggplot(sml_muestral, aes(x = Beta)) +
  geom_hline(yintercept = 0, color = "gray", linetype = "dotted") +
  geom_abline(intercept = 0, slope = prima_muestral, color = "#d62728", linewidth = 1) +
  geom_segment(aes(xend = Beta, y = Retorno_SML, yend = Retorno_medio),
               linetype = "dashed", color = "gray40") +
  geom_point(aes(y = Retorno_medio), color = "steelblue", size = 3) +
  geom_text(aes(y = Retorno_medio, label = sprintf("%s (α = %+.1f%%)", Accion, Alfa_anual * 100)),
            hjust = -0.1, vjust = -0.6) +
  geom_point(data = tibble(Beta = 1, Retorno = prima_muestral), aes(y = Retorno),
             shape = 23, fill = "gold", color = "black", size = 3.5) +
  scale_x_continuous(expand = expansion(mult = c(0.02, 0.3))) +
  scale_y_continuous(labels = percent_format(accuracy = 1)) +
  expand_limits(x = 0) +
  labs(x = "Beta (retornos en exceso)", y = "Retorno en exceso promedio (anualizado)",
       title = "El alfa de Jensen como distancia a la SML muestral",
       subtitle = "Recta roja: SML muestral. Rombo dorado: el mercado (β = 1)") +
  theme_minimal()

### 🧭 Síntesis

* Con **retornos brutos**, la pendiente es el beta, pero el intercepto **no** tiene una interpretación económica clara: mezcla el alfa con $\left(1-\beta_{i}\right)r_{f}$.
* Con **retornos en exceso**, la pendiente sigue siendo el beta, y el intercepto es el **alfa de Jensen**: el retorno anormal ajustado por riesgo, o la distancia de la acción a la SML.
* El **beta** es el insumo para calcular el **costo del capital propio** con el CAPM. El **alfa**, en cambio, sirve para **evaluar el desempeño** de una acción o de un administrador de portafolios, una vez descontado el riesgo sistemático que asumió.
* Bajo el CAPM, el alfa esperado es cero. Con muestras pequeñas, los alfas estimados son ruidosos, y pocas veces son estadísticamente distintos de cero.